In [25]:
import os
import json
import requests


transcripts = {}
for i in range(8):
    transcripts[i+1] = {}


def get_transcript(call_id):
    headers = { 'Authorization': os.getenv('BLAND_AUTH_TOKEN') }
    response = requests.request("GET", f"https://api.bland.ai/v1/calls/{call_id}", headers=headers, verify=False)
    response_dict = json.loads(response.text)

    return response_dict["concatenated_transcript"]


def set_transcripts(participant_id, call_ids):
    for i, call_id in enumerate(call_ids, 1):
        transcripts[participant_id][f'{i}_{call_id}'] = get_transcript(call_id)


In [20]:
participants = {
    1:
        [
            'm', 'nonnative', 
            ["236c75ef-31f0-41a5-9e04-6699c121a94e", "1ff0a420-77ba-46b1-9873-f8b29c05aa41", "f311bb3e-75a7-4300-95f5-a7afb8fd29a5", 
            "0e33d383-5e2c-47b0-ace4-a3efe5f966cb", "94441d4f-f4e1-4156-9a7e-342fb123d166"]
        ],
    2:
        [
            'm', 'native', 
            ["0a6bc04e-2b15-4e12-95ea-9e590d377e95", "fed90af9-0475-46aa-b7f5-1d332a6f6b23", "945ad965-8cc1-4f7a-9a4e-bbd458aa587c", 
            "f4b0d4bf-62ef-4716-8488-ab062501f81e", "f82df6f2-d6f4-44de-85a0-c5359baf8db7"]
        ],
    3:
        [
            'f', 'nonnative', 
            ['46b05daf-6232-4aca-88ce-b284df24e67e', 'c2edfc21-c4e6-4f3a-8d9f-bc2f895f7123', '6b39b9ad-605d-4b4c-91fa-8cc90da45943',
            'e7c63a0f-41c0-48c7-a851-721e90f0635a', '5aeae7d7-fa9f-4773-9393-434cbdb612d8']
        ],
    4:
        [
            'm', 'native', 
            ["dbfb95a3-b299-482b-89e4-969242446167", "34b3ab14-4207-4759-a159-08238255c7db", "8d6fedcc-c02b-47c3-b21e-f033f494b80b", 
             "1c1e55b6-61b2-4b64-a297-f3625d562a19", "017f2502-46bb-44a7-80d3-0d9a9ed04afe"]
        ]
}


In [26]:
for id, [sex, native, call_ids] in participants.items():
    if id != 4:
        continue
    set_transcripts(id, call_ids)

/Users/dobbinsnj/work/ai-agent-based-survey/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.bland.ai'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dobbinsnj/work/ai-agent-based-survey/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.bland.ai'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dobbinsnj/work/ai-agent-based-survey/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.bland.ai'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usa

In [30]:
for participant_id, values in transcripts.items():
    
    if participant_id != 4:
        continue

    sex, native, _ = participants[participant_id]

    for call_id, transcript in values.items():
        directory = os.path.join('./call-data', 'calls', f'{participant_id}_{sex}_{native}_speaker')
        if not os.path.exists(directory):
            os.mkdir(directory)

        if not os.path.exists(os.path.join(directory, f'call_{call_id}.txt')):
            with open(os.path.join(directory, f'call_{call_id}.txt'), 'w+', encoding='utf-8') as fout:
                fout.write(transcript)
            with open(os.path.join(directory, f'call_{call_id}_corrected.txt'), 'w+', encoding='utf-8') as fout:
                fout.write(transcript)

In [17]:
import os
import sacrebleu
from rouge_score import rouge_scorer


scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
data_path = os.path.join('./call-data', 'calls')
results = {}

for dir in os.listdir(data_path):
    participant_id, sex, speaker, _ = dir.split('_')
    results[participant_id] = []
    transcripts = [x for x in os.listdir(os.path.join(data_path, dir)) if 'corrected' not in x]

    for transcript_path in transcripts:
        with open(os.path.join(data_path, dir, transcript_path), 'r', encoding='utf-8') as fin:
            original = fin.read()
        with open(os.path.join(data_path, dir, transcript_path.replace('.txt', '_corrected.txt')), 'r', encoding='utf-8') as fin:
            corrected = fin.read()

        #score = sacrebleu.raw_corpus_bleu(original, [corrected], 0.0).score / 100
        score = scorer.score(original, corrected)
        results[participant_id].append(score['rougeL'])

In [18]:
results

{'3': [Score(precision=0.9940357852882704, recall=0.9970089730807578, fmeasure=0.9955201592832256),
  Score(precision=0.9874213836477987, recall=0.98125, fmeasure=0.9843260188087775),
  Score(precision=0.9882005899705014, recall=0.9896602658788775, fmeasure=0.9889298892988929),
  Score(precision=0.995850622406639, recall=0.9948186528497409, fmeasure=0.9953343701399688),
  Score(precision=0.9901423877327492, recall=0.9923161361141603, fmeasure=0.9912280701754387)],
 '2': [Score(precision=1.0, recall=1.0, fmeasure=1.0),
  Score(precision=0.9976851851851852, recall=0.9976851851851852, fmeasure=0.9976851851851852),
  Score(precision=0.9924178601516428, recall=0.9932546374367622, fmeasure=0.9928360724820902),
  Score(precision=1.0, recall=0.9989071038251366, fmeasure=0.9994532531437945),
  Score(precision=0.9904357066950054, recall=0.9883351007423118, fmeasure=0.9893842887473461)],
 '1': [Score(precision=0.9928664072632944, recall=0.9935107073329007, fmeasure=0.9931884528057087),
  Score(pr